# Quote -> WhatsApp

Fetch the live quote for RELIANCE from OpenAlgo and send a formatted snapshot to the operator's WhatsApp.

**Prerequisites**

1. WhatsApp device is already paired in OpenAlgo (visit `/whatsapp` in the web UI, click **Start pairing**, and scan the QR with your phone).
2. `openalgo >= 1.0.50` is installed (`pip install -U openalgo`). The new `client.whatsapp(...)` method ships in 1.0.50.
3. The OpenAlgo server is running on `http://127.0.0.1:5000` (adjust the `host` argument if yours runs elsewhere).

**Note on API key**

The key embedded below is for local development against this exact OpenAlgo instance. Replace with your own key from the **API Key** page in the OpenAlgo dashboard before sharing or committing this notebook.

In [1]:
from openalgo import api

# Replace with your own API key from the OpenAlgo /apikey page.
client = api(
    api_key="0b2b621a141bfeffedeee64b1a5414a20d5f073916ca6f8cf2202b885bec216c",
    host="http://127.0.0.1:5000",
)

## 1. Fetch the RELIANCE quote

`client.quotes(symbol, exchange)` returns the standard OpenAlgo quote payload — `open`, `high`, `low`, `ltp`, `bid`, `ask`, `prev_close`, `volume`.

In [2]:
symbol = "RELIANCE"
exchange = "NSE"

response = client.quotes(symbol=symbol, exchange=exchange)
response

{'data': {'ask': 0,
  'bid': 0,
  'high': 1364.8,
  'low': 1329.2,
  'ltp': 1336.4,
  'oi': 0,
  'open': 1356.8,
  'prev_close': 1361.8,
  'volume': 0},
 'status': 'success'}

## 2. Format the WhatsApp message

WhatsApp renders `*bold*` markup as bold text on the recipient's phone, so we use that for the symbol header. The body lists the headline price stats compactly.

In [3]:
if response.get("status") != "success":
    raise RuntimeError(f"Quote fetch failed: {response.get('message')}")

data = response["data"]
change = data["ltp"] - data["prev_close"]
pct = (change / data["prev_close"]) * 100 if data["prev_close"] else 0.0

message = (
    f"*{symbol} {exchange}*\n"
    f"LTP:       {data['ltp']}\n"
    f"Change:    {change:+.2f} ({pct:+.2f}%)\n"
    f"Open:      {data['open']}\n"
    f"High:      {data['high']}\n"
    f"Low:       {data['low']}\n"
    f"Bid / Ask: {data['bid']} / {data['ask']}\n"
    f"Volume:    {data['volume']:,}"
)
print(message)

*RELIANCE NSE*
LTP:       1336.4
Change:    -25.40 (-1.87%)
Open:      1356.8
High:      1364.8
Low:       1329.2
Bid / Ask: 0 / 0
Volume:    0


## 3. Send the snapshot to yourself on WhatsApp

`client.whatsapp(text)` with no recipient argument routes to the paired device's own number — the operator. Returns a per-recipient delivery report by default (`wait_for_delivery=True`).

In [4]:
result = client.whatsapp(message)
result

{'data': {'failed': [], 'sent': ['<self>'], 'skipped': 0},
 'message': 'Delivered to 1, failed 0',
 'status': 'success'}

## 4. (Optional) Send to a specific phone number

Pass an E.164 digit string in `to=...`. Up to 5 numbers can be passed as a list for a small broadcast (the server caps it there as a WhatsApp ToS-safety guardrail).

In [ ]:
# Uncomment and set a real number to test.
# result = client.whatsapp(message, to="919876543210")
# result